In [0]:
from pyspark.sql.functions import col, sum, count, when

In [0]:
catalog_name = "ecommerce_catalog"
schema_name = "ecom_schema"

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

df_full_data = spark.table("silver_full_data")
df_silver_sales = spark.table("silver_sales")
df_silver_customers = spark.table("silver_customers")

print(" Tables loaded successfully.")

In [0]:

df_gold_segments = df_full_data.withColumn(
    "customer_segment",
    when(col("total_spent") >= 5000, "Platinum")
    .when((col("total_spent") >= 2000) & (col("total_spent") < 5000), "Gold")
    .otherwise("Silver")
)
df_gold_segments.write.format("delta").mode("overwrite").saveAsTable("gold_customer_segments")
print(" gold_customer_segments saved.")

In [0]:
df_gold_monthly_sales = df_silver_sales.groupBy("sales_year", "sales_month").agg(
    sum("amount").alias("monthly_revenue"),
    count("transaction_id").alias("monthly_orders")
).orderBy("sales_year", "sales_month")

df_gold_monthly_sales.write.format("delta").mode("overwrite").saveAsTable("gold_monthly_sales_trend")
print(" 'gold_monthly_sales_trend' successfully saved!")
display(df_gold_monthly_sales)

In [0]:
df_sales_with_country = df_silver_sales.join(
    df_silver_customers, 
    on="customer_id", 
    how="inner"
)
# 2. Country-wise aggregation
df_gold_country_performance = df_sales_with_country.groupBy("country").agg(
    sum("amount").alias("total_revenue"),
    count("transaction_id").alias("total_transactions")
).orderBy(col("total_revenue").desc())
# 3. Gold Table save
df_gold_country_performance.write.format("delta").mode("overwrite").saveAsTable("gold_country_performance")

print("✅ Gold Table successfully saved!")
display(df_gold_country_performance)

#### Segment-wise Revenue (Marketing view)

In [0]:
%sql
SELECT 
    customer_segment, 
    COUNT(customer_id) as total_customers,
    SUM(total_spent) as total_revenue,
    AVG(total_spent) as avg_spend_per_customer
FROM gold_customer_segments
GROUP BY customer_segment
ORDER BY total_revenue DESC;

#### Top 3 Countries by Sales

In [0]:
%sql
SELECT 
    country, 
    total_revenue, 
    total_transactions
FROM gold_country_performance
ORDER BY total_revenue DESC
LIMIT 3;

#### Month-on-Month Growth (Business Trend)

In [0]:
%sql
SELECT 
    sales_year, 
    sales_month, 
    monthly_revenue,
    LAG(monthly_revenue) OVER (ORDER BY sales_year, sales_month) as previous_month_revenue
FROM gold_monthly_sales_trend;